In [1]:
from dotenv import load_dotenv

load_dotenv()

True

### 一个完整的Agent至少要包含两个关键部分：
- 模型
- 工具  

定义带有工具的Agent的基本流程如下：
- 定义工具
- 初始化模型
- 初始化Agent，绑定模型和工具

###  1.自定义工具
工具，本质是一个可调用的函数，但是函数是给模型调用的。因此除了定义函数外，还需要清晰描述这个工具，让模型知道这个工具如何使用。  
包括下列信息：  
- 工具名
- 工具的作用
- 工具需要的参数

模型如何知道工具信息：  
Agent在接收到用户的提问后，会将用户问题（message）、工具信息（tools）组装成请求参数一起发送给模型，由模型来分析问题和分析工具的调用

#### 1.1.基于tool描述工具
在LangChain中，定义工具需要用到@tool装饰器，我们可以通过装饰器来定义工具名、工具的作用

In [2]:
from langchain_core.tools import tool

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

### 1.2.使用函数名和文档注释描述工具
如果不@tool装饰器没有定义工具名和作用描述，此时：
- 工具名：默认就是函数名
- 工具所需的参数：默认就是函数的参数列表
- 工具作用的描述：默认就是函数的文档注释

In [3]:
from langchain_core.tools import tool

# 通过tool装饰器定义工具
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
# 定义一个查询天气的tool
@tool
def get_weather(location:str, units:str="celsius", include_forecast:bool = False) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

### 1.3.定义Pydantic Model描述参数
如果函数的参数比较多，而且比较复杂，此时建议通过pydantic model来描述参数列表

In [5]:
# 通过自定义model来约束入参
from pydantic import BaseModel, Field
from typing import Literal

# 一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数"""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location:str, units:str="celsius", include_forecast:bool = False) -> str:
    """
    Get current weather and optional forecast.
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

工具调用方式与普通函数调用方式一致

In [6]:
square_root.invoke({"x": 467})

21.61018278497431

In [7]:
get_weather.invoke({"location": "杭州", "include_forecast": True})

'Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny'

### 测试

In [8]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model = "deepseek-chat",
    tools=[square_root, get_weather]
)

In [9]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="杭州接下来几天天气如何？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

好的，我来查询杭州的天气情况，包括未来几天的预报。Current weather in 杭州: 22 degrees C
Next 5 days: Sunny杭州接下来几天的天气情况如下：

### ☀️ 当前天气
- **温度**：22°C
- **天气状况**：晴朗

### 📅 未来5天天气预报
- **整体趋势**：未来几天都是 **晴朗（Sunny）** 的好天气！

也就是说，杭州接下来几天天气非常不错，阳光明媚，适合出行和户外活动。不过昼夜温差可能需要注意，建议带一件薄外套。

请问还需要了解其他城市的天气吗？😊

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="467和529的平方根是多少？")]},
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

467和529的平方根是多少？
None
================================== Ai Message ==================================

好的，我来分别计算467和529的平方根。
Tool Calls:
  square_root (call_00_OpemeussnniATV2qeyqQ7282)
 Call ID: call_00_OpemeussnniATV2qeyqQ7282
  Args:
    x: 467
  square_root (call_01_DYJavurTXeLQIlUeulAT5362)
 Call ID: call_01_DYJavurTXeLQIlUeulAT5362
  Args:
    x: 529
None
================================= Tool Message =================================
Name: square_root

21.61018278497431
None
================================= Tool Message =================================
Name: square_root

23.0
None
================================== Ai Message ==================================

计算结果如下：

- **467的平方根** ≈ **21.6102**
- **529的平方根** = **23**（恰好是整数，因为 23 × 23 = 529）
None


## 预定义工具

In [ ]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_result = 5,
    topic = "general"
)

search_tool.invoke(".......")

In [ ]:
# 创建智能体，使用预定义工具tavily
agent = create_agent(
    model="deepseek-chat",
    tools=[search_tool],
    system_prompt="..........."
)

response = agent.invoke(
    {"messages": [HumanMessage(content=".............")]}
)

for message in response["messages"]:
    message.pretty_print()